# Running Tonic with Its Arms Separated

Section 7 of the paper measures $\psi$, the captured heaviness mass, and calls it a **motivated
proxy** for the predictor channel in Tonic's family. It is a proxy because we never ran their
algorithm. This notebook closes that gap.

Their own experiments already contain the three arms we need; nothing new has to be implemented. We
only need the **raw estimates** rather than the reported relative errors, so that variances can be
computed:

| arm | what it is | channel |
|---|---|---|
| `wrs` | WRS, no predictor | unaugmented baseline |
| `tonic_mindeg` | Tonic + `MinDegreePredictor` | free statistic (degree, one pass) |
| `tonic_oracle` | Tonic + `OracleExact` | exact heaviness oracle |

All three are unbiased (their Thm 4.1), so the sample variance over repeated trials gives

$$G_\mathrm{stat}=\frac{\mathrm{Var}[\textsf{wrs}]}{\mathrm{Var}[\textsf{tonic\_mindeg}]},\qquad
G_\mathrm{pred}'=\frac{\mathrm{Var}[\textsf{tonic\_mindeg}]}{\mathrm{Var}[\textsf{tonic\_oracle}]},\qquad
G_\mathrm{total}=G_\mathrm{stat}\cdot G_\mathrm{pred}'.$$

The telescoping is exact by construction, so it is a consistency check, not a result. **The result
is the size of $G_\mathrm{pred}'$** --- the paper claims it is small.

### Optional fourth arm

If the CLI accepts $\alpha=0$ (no waiting room), then WRS at $\alpha=0$ versus $\alpha=0.05$
isolates the temporal-locality channel, which is the localization channel of Definition 3.5 in
their setting. Chen et al. (2022, App. G) describe the waiting room as "an oracle predicting recent
edges as heavy"; this measures what that is worth. If it works, all three channels are measured on
a deployed algorithm.

### An honest warning

**This notebook has not been tested against the actual Tonic CLI.** It was written from their
paper, not from their repository. Section 2 prints the real interface; fill in `CONFIG` from what
it shows, then continue. Everything downstream of `CONFIG` is generic and tested.

The most likely thing to break is `ESTIMATE_REGEX`. The second most likely is the predictor file
format. Section 5 is a smoke test that catches both before the full run.

### Either outcome is publishable

A small $G_\mathrm{pred}'$ with a large $G_\mathrm{stat}$ confirms the paper's transfer claim,
measured end to end rather than inferred. A large $G_\mathrm{pred}'$ refutes it, and Section 7 gets
rewritten as a negative result. Report whichever happens.

## 1. Environment

In [ ]:
import os, re, json, shutil, subprocess, sys, time, gzip, urllib.request
import numpy as np
print(sys.version.split()[0])

ROOT = "/content/tonic_work" if os.path.isdir("/content") else os.path.abspath("tonic_work")
SRC  = os.path.join(ROOT, "Tonic")
DATA = os.path.join(ROOT, "data")
os.makedirs(DATA, exist_ok=True)

def sh(cmd, cwd=None, check=False, quiet=False):
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if not quiet and (p.stdout or p.stderr):
        print((p.stdout + p.stderr)[:4000])
    if check and p.returncode != 0:
        raise SystemExit(f"failed: {cmd}")
    return p

sh("cmake --version | head -1; g++ --version | head -1; git --version", quiet=False)

## 2. Clone, build, and print the real interface

**Send the output of this section back before running anything else** — the `CONFIG` block in
Section 3 can then be filled in exactly instead of guessed.

In [ ]:
os.makedirs(ROOT, exist_ok=True)
if not os.path.isdir(SRC):
    sh("git clone --depth 1 https://github.com/VandinLab/Tonic", cwd=ROOT, check=True)

print("=" * 70, "\nREPOSITORY LAYOUT\n", "=" * 70)
sh("find . -not -path './.git/*' -type f | head -60", cwd=SRC)

print("=" * 70, "\nREADME (first 100 lines)\n", "=" * 70)
for f in ("README.md", "readme.md", "README"):
    p = os.path.join(SRC, f)
    if os.path.exists(p):
        print("".join(open(p, encoding="utf-8", errors="ignore").readlines()[:100]))
        break

print("=" * 70, "\nBUILD\n", "=" * 70)
if os.path.exists(os.path.join(SRC, "CMakeLists.txt")):
    sh("mkdir -p build && cd build && cmake .. && make -j4", cwd=SRC)
elif os.path.exists(os.path.join(SRC, "Makefile")):
    sh("make -j4", cwd=SRC)
else:
    print("no CMakeLists.txt or Makefile at top level -- look at the layout above")

print("=" * 70, "\nEXECUTABLES\n", "=" * 70)
exes = sh("find . -type f -perm -u+x -not -path './.git/*' -not -name '*.sh' "
          "-not -name '*.py' -not -name '*.txt'", cwd=SRC, quiet=True).stdout.split()
print("\n".join(exes) or "(none found)")

print("=" * 70, "\nHELP TEXT\n", "=" * 70)
for b in exes[:4]:
    for flag in ("--help", "-h", ""):
        h = sh(f"{b} {flag}", cwd=SRC, quiet=True)
        txt = (h.stdout + h.stderr).strip()
        if txt:
            print(f"$ {b} {flag}\n{txt[:2000]}\n" + "-" * 60); break

## 3. CONFIG — fill this in from Section 2

Everything the notebook needs to know about their interface lives here.

In [ ]:
# =============================================================================
#  CONFIG -- final, from the observed CLI and one real run.
#
#  Observed output of a successful invocation:
#     Edge Oracle successfully read in time 0.001!
#     Size of the oracle = 1448 edges
#     Starting Tonic Algo - alpha 0.050, beta = 0.001 | Memory Budget = 1448
#     WR size = 72, H size = 1, SL size = 1375
#     TonicINS Algo successfully run in time 0.013!
#     Estimated count T = 46821.600337
#
#  Two corrections follow:
#   * the output FILE is never created; the estimate is on stdout.  We therefore
#     anchor the regex on "Estimated count T =" instead of taking the largest
#     number, which would collide with Memory Budget / SL size on graphs where
#     T is of the same order (p2p: T ~ 2384, k = 2077).
#   * the beta probe showed sd FALLING monotonically as beta -> 0 (4274, 3711,
#     2133, 1353), so a small beta is NOT an inert predictor and NOT a baseline.
#     The wrs arm is dropped.  G_pred' needs no baseline, and it is the quantity
#     Section 7 actually requires.
# =============================================================================

BIN  = "build/Tonic"
FLAG = 0
TAIL = ""
BETA_FULL = 0.20

CONFIG = dict(
    arms = {
        "tonic_mindeg": (BETA_FULL, "nodes", "mindeg"),   # free statistic
        "tonic_oracle": (BETA_FULL, "edges", "oracle"),   # exact heaviness oracle
    },
    k_fraction = 0.10,
    alpha      = 0.05,
    trials     = 500,        # 50 leaves every CI straddling 1; they use 500 too
)

ESTIMATE_REGEX = r"Estimated\s+count\s+T\s*=\s*([0-9]+\.?[0-9]*(?:[eE][+-]?[0-9]+)?)"

print("arms:", list(CONFIG["arms"]), "| trials:", CONFIG["trials"])
print("G_pred' = Var[tonic_mindeg] / Var[tonic_oracle]  -- no baseline arm needed.")

## 4. Datasets and predictor files

The same six SNAP graphs as the paper, so the numbers are directly comparable. We also build the
two predictor files:

* **OracleExact** — exact per-edge triangle count $\Delta(e)$;
* **MinDegreePredictor** — node degrees, from which $\min\{d(u),d(v)\}$ is formed.

Their paper stores the top $10\%$ of edges by predicted heaviness for the oracle, and the highest
degree nodes for the degree predictor. We write both full and top-$10\%$ variants; **check which
format their loader expects** and set `PRED_STYLE` accordingly.

In [ ]:
SNAP = {
    "ca-GrQc":        "https://snap.stanford.edu/data/ca-GrQc.txt.gz",
    "ca-HepTh":       "https://snap.stanford.edu/data/ca-HepTh.txt.gz",
    "p2p-Gnutella08": "https://snap.stanford.edu/data/p2p-Gnutella08.txt.gz",
    "oregon1":        "https://snap.stanford.edu/data/oregon1_010331.txt.gz",
    "fb-ego":         "https://snap.stanford.edu/data/facebook_combined.txt.gz",
    "email-Enron":    "https://snap.stanford.edu/data/email-Enron.txt.gz",
}
PRED_STYLE = "top10"     # "top10" (their default) or "full"

def fetch(name, url):
    out = os.path.join(DATA, name + ".txt")
    if os.path.exists(out): return out
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as r:
        txt = gzip.decompress(r.read()).decode("utf-8", "ignore")
    seen = set(); lines = []
    for line in txt.splitlines():
        if line and line[0] != "#":
            p = line.split()
            if len(p) >= 2 and p[0] != p[1]:
                a, b = int(p[0]), int(p[1])
                e = (a, b) if a < b else (b, a)
                if e not in seen: seen.add(e); lines.append(f"{e[0]} {e[1]}")
    open(out, "w").write("\n".join(lines) + "\n")
    return out

def build_predictors(name, path):
    E = [tuple(map(int, l.split())) for l in open(path) if l.strip()]
    nodes = sorted({x for e in E for x in e}); idx = {v: i for i, v in enumerate(nodes)}
    adj = [set() for _ in nodes]
    for u, v in E: adj[idx[u]].add(idx[v]); adj[idx[v]].add(idx[u])
    deg = [len(a) for a in adj]
    delta = []
    for u, v in E:
        a, b = adj[idx[u]], adj[idx[v]]
        if len(a) > len(b): a, b = b, a
        delta.append(sum(1 for w in a if w in b))
    order = sorted(range(len(E)), key=lambda i: -delta[i])
    keep = order[:max(1, len(E) // 10)] if PRED_STYLE == "top10" else order
    po = os.path.join(DATA, f"{name}_oracle.txt")
    with open(po, "w") as f:
        for i in keep: f.write(f"{E[i][0]} {E[i][1]} {delta[i]}\n")
    nord = sorted(range(len(nodes)), key=lambda i: -deg[i])
    uniq = {x for i in keep for x in E[i]}
    nkeep = nord[:len(uniq)] if PRED_STYLE == "top10" else nord
    pm = os.path.join(DATA, f"{name}_mindeg.txt")
    with open(pm, "w") as f:
        for i in nkeep: f.write(f"{nodes[i]} {deg[i]}\n")
    return po, pm, len(E), sum(delta) // 3

DATASETS = []
for name, url in SNAP.items():
    try:
        t0 = time.time(); p = fetch(name, url)
        po, pm, m, T = build_predictors(name, p)
        DATASETS.append((name, p, m, T))
        print(f"  OK   {name:16s} m={m:8d} T={T:9d}   ({time.time()-t0:.1f}s)")
    except Exception as ex:
        print(f"  FAIL {name:16s} {type(ex).__name__}: {ex}")

print("\nfirst lines of one oracle file (check this against their loader):")
sh(f"head -3 {os.path.join(DATA,'ca-GrQc_oracle.txt')}")
print("first lines of one degree file:")
sh(f"head -3 {os.path.join(DATA,'ca-GrQc_mindeg.txt')}")

## 5. Smoke test — one graph, five trials

Run this before the full sweep. It catches a broken command template, a wrong regex, and a
predictor file their loader rejects, in about ten seconds instead of after an hour.

In [ ]:
OUTDIR = os.path.join(ROOT, "out"); os.makedirs(OUTDIR, exist_ok=True)

def invoke(name, edges, m, seed, beta, otype, pkey, echo=False, tag="run"):
    pred = os.path.join(DATA, f"{name}_{pkey}.txt")
    outp = os.path.join(OUTDIR, f"{name}_{tag}_{seed}.txt")
    k = int(CONFIG["k_fraction"] * m)
    cmd = (f"{BIN} {FLAG} {seed} {k} {CONFIG['alpha']} {beta} {edges} "
           f"{pred} {otype} {outp}{TAIL}").strip()
    r = subprocess.run(cmd, shell=True, cwd=SRC, capture_output=True, text=True)
    blob = r.stdout + r.stderr
    if os.path.exists(outp):
        blob += "\n" + open(outp, errors="ignore").read()
    if echo:
        print(f"$ {cmd}\n[exit {r.returncode}]\n{blob[:900]}\n" + "-"*60)
    hit = re.search(ESTIMATE_REGEX, blob)
    return (float(hit.group(1)) if hit else None), blob

def one_run(arm, name, edges, m, seed, echo=False):
    beta, otype, pkey = CONFIG["arms"][arm]
    return invoke(name, edges, m, seed, beta, otype, pkey, echo=echo, tag=arm)

# the anchored regex must find the estimate and nothing else
_v, _b = invoke(DATASETS[0][0], DATASETS[0][1], DATASETS[0][2], 0, 0.2,
                "edges", "oracle", echo=True, tag="regex_check")
assert _v is not None, "anchored regex failed -- read the echo above"
print(f"parsed estimate = {_v}   (true T = {DATASETS[0][3]})")
print("helpers defined.")

In [ ]:
name, edges, m, T = DATASETS[0]
print(f"smoke test on {name}  (m={m}, true T={T})\n")
ok = True
for arm in CONFIG["arms"]:
    vals = []
    for s in range(8):
        v, _ = one_run(arm, name, edges, m, s)
        if v is None:
            print(f"  !! {arm}: nothing parsed"); ok = False; break
        vals.append(v)
    if vals:
        print(f"  {arm:16s} mean={np.mean(vals):12.1f} sd={np.std(vals, ddof=1):10.1f} "
              f"rel.bias={(np.mean(vals)-T)/T:+.4f}")
print("\nSMOKE TEST", "PASSED" if ok else "FAILED")
print("Both arms should be unbiased (rel.bias near 0). Variances are not")
print("interpretable at 8 trials -- that is what the full run is for.")

## 6. Full run

In [ ]:
if not ok:
    raise SystemExit(
        "The smoke test failed, so the full sweep would collect nothing.\n"
        "Fix CONFIG / ESTIMATE_REGEX first, or run tonic_diagnose.ipynb.")

RAW = {}
for name, edges, m, T in DATASETS:
    RAW[name] = {"m": m, "T": T, "arms": {}}
    for arm in [a for a, v in CONFIG["arms"].items() if v]:
        t0 = time.time(); vals = []
        for s in range(CONFIG["trials"]):
            v, _ = one_run(arm, name, edges, m, s)
            if v is not None: vals.append(v)
        if not vals:
            raise SystemExit(
                f"No estimate parsed for {name}/{arm}. Nothing downstream can work.\n"
                "Run tonic_diagnose.ipynb and fix CONFIG before continuing.")
        RAW[name]["arms"][arm] = vals
        print(f"  {name:16s} {arm:16s} n={len(vals):4d} mean={np.mean(vals):12.1f} "
              f"sd={np.std(vals, ddof=1):10.1f}  [{time.time()-t0:.1f}s]")
    print()
json.dump(RAW, open(os.path.join(ROOT, "raw_estimates.json"), "w"), indent=1)
print("saved raw_estimates.json")

## 7. The decomposition

Variance ratios with bootstrap confidence intervals. A ratio of two variances is skewed, so a
normal approximation would misstate the interval; we resample instead.

In [ ]:
def boot_ratio(a, b, reps=6000, seed=0):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a, float), np.asarray(b, float)
    r = [np.var(rng.choice(a, len(a)), ddof=1) / np.var(rng.choice(b, len(b)), ddof=1)
         for _ in range(reps)]
    return (float(np.var(a, ddof=1) / np.var(b, ddof=1)),
            float(np.percentile(r, 2.5)), float(np.percentile(r, 97.5)))

DEC = {}
print(f"{'graph':16s} {'G_pred(residual)':>26s} {'excludes 1?':>12s} {'psi share':>10s}")
PSI = {"ca-GrQc": 0.993, "fb-ego": 0.947, "oregon1": 0.865,
       "p2p-Gnutella08": 0.798, "email-Enron": 0.769, "ca-HepTh": 0.539}
for name, d in RAW.items():
    A = d["arms"]
    if not {"tonic_mindeg", "tonic_oracle"} <= set(A): continue
    gp, lo, hi = boot_ratio(A["tonic_mindeg"], A["tonic_oracle"])
    DEC[name] = dict(G_pred=gp, ci=[lo, hi], n=len(A["tonic_oracle"]))
    print(f"{name:16s} {gp:8.3f}  [{lo:6.3f}, {hi:6.3f}] {str(lo > 1 or hi < 1):>12s} "
          f"{PSI.get(name, float('nan')):10.3f}")
    for arm, e in A.items():
        bias = (np.mean(e) - d["T"]) / d["T"]
        if abs(bias) > 0.02:
            print(f"    WARNING {arm}: rel. bias {bias:+.4f}")
json.dump(DEC, open(os.path.join(ROOT, "decomposition.json"), "w"), indent=1)
print("\nG_pred' is the residual prediction channel: how much an EXACT heaviness")
print("oracle buys over a free degree score, inside a deployed algorithm.")
print("Values near 1 are the paper's claim. A CI straddling 1 means the run")
print("could not resolve the effect -- report that, do not read a point estimate.")

## 8. Report block — copy this back

In [ ]:
print("=" * 78); print("TONIC THREE-ARM DECOMPOSITION"); print("=" * 78)
print(json.dumps({"config": {k: v for k, v in CONFIG.items() if k != "arms"},
                  "pred_style": PRED_STYLE,
                  "raw_summary": {n: {a: dict(n=len(v), mean=round(float(np.mean(v)), 2),
                                              sd=round(float(np.std(v, ddof=1)), 2))
                                      for a, v in d["arms"].items()}
                                  for n, d in RAW.items()},
                  "decomposition": {n: {k: (round(v, 5) if isinstance(v, float)
                                            else [round(x, 5) for x in v])
                                        for k, v in r.items()}
                                    for n, r in DEC.items()}}, indent=1))
print("-" * 78)
gp = [r["G_pred"] for r in DEC.values()]
gs = [r["G_stat"] for r in DEC.values()]
if gp:
    print(f"G_stat  across instances: [{min(gs):.3f}, {max(gs):.3f}]")
    print(f"G_pred' across instances: [{min(gp):.3f}, {max(gp):.3f}]")
    print("small G_pred' with large G_stat  => Section 7's transfer claim, measured")
    print("large G_pred'                    => transfer refuted; report it as such")
print("=" * 78)